In [ ]:
# ============================================
# DFD01 T=32 — PREPROCESSING WITH RESUME
# ============================================
# Input: "dfdc-dataset-02-dfd-01" (only DFD01 part is used)
# Output: preprocessed_DFD01_32.tar.gz
#
# RESUME SUPPORT:
# If a previous run timed out (e.g., processed 2609/3431),
# add the PARTIAL output as a Kaggle dataset input.
# This notebook will restore it and continue from where it stopped.
#
# Steps:
# 1. Check if partial output exists in /kaggle/input/
# 2. If yes, copy it to /kaggle/working/ (restoring progress)
# 3. Run preprocess_videos.py which auto-skips existing videos
# 4. Package final result
#
# IMPORTANT: Pillow must be downgraded BEFORE other imports

NUM_FRAMES = 32
OUTPUT_SIZE = 224
MIN_FACE_CONF = 0.90
MIN_DETECTION_RATIO = 0.55
DETECTOR_MAX_SIDE = 960

import subprocess, sys, os, time, shutil, glob

# Step 1: Force downgrade Pillow FIRST
print('=== Downgrading Pillow ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'],
               check=True)
print('Pillow downgraded.')

# Step 2: Install other dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch',
                'opencv-python-headless', 'tqdm'],
               check=True)
print('Dependencies installed.')

# Step 3: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 4: RESUME — restore partial output from previous run
output_name = f'preprocessed_DFD01_{NUM_FRAMES}'
output_path = f'/kaggle/working/{output_name}'

# ==============================================
# DEBUG: Show FULL contents of /kaggle/input/
# This helps diagnose resume failures
# ==============================================
print('\n=== DEBUG: /kaggle/input/ contents ===')
for entry in sorted(os.listdir('/kaggle/input')):
    entry_path = f'/kaggle/input/{entry}'
    if os.path.isdir(entry_path):
        try:
            sub = sorted(os.listdir(entry_path))
            print(f'  {entry}/ -> {sub}')
            # Go one level deeper for each subdirectory
            for s in sub:
                s_path = os.path.join(entry_path, s)
                if os.path.isdir(s_path):
                    try:
                        sub2 = sorted(os.listdir(s_path))[:10]
                        print(f'    {s}/ -> {sub2}')
                    except Exception as e2:
                        print(f'    {s}/ -> ERROR: {e2}')
        except Exception as e:
            print(f'  {entry}/ -> ERROR: {e}')
    else:
        print(f'  {entry} (file)')
print(f'=== DEBUG: Looking for directory named "{output_name}" ===\n')

# Look for partial output uploaded as Kaggle dataset input
# The actual slug depends on how the dataset was named when saved
partial_candidates = [
    f'/kaggle/input/dfd01-preprocessed-t32-partial/{output_name}',
    f'/kaggle/input/preprocessed-dfd01-32/{output_name}',
    f'/kaggle/input/preprocessed-dfd01-32-partial/{output_name}',
    f'/kaggle/input/{output_name}/{output_name}',
    f'/kaggle/input/dfd01-32-partial/{output_name}',
]

# Also auto-scan /kaggle/input/ for any directory containing our output_name
for entry in os.listdir('/kaggle/input'):
    candidate = f'/kaggle/input/{entry}/{output_name}'
    if candidate not in partial_candidates and os.path.isdir(candidate):
        partial_candidates.insert(0, candidate)
        print(f'[AUTO-DETECT] Found partial output candidate: {candidate}')

# Print all candidates and whether they exist
print('=== Resume candidates check ===')
for i, candidate in enumerate(partial_candidates):
    exists = os.path.isdir(candidate)
    print(f'  [{i}] {candidate} -> EXISTS={exists}')
print()

restored = False
for candidate in partial_candidates:
    if os.path.isdir(candidate):
        print(f'\n=== RESUME: Found partial output at {candidate} ===')
        # Count existing videos
        existing_real = len(glob.glob(os.path.join(candidate, 'real', '*/')))
        existing_fake = len(glob.glob(os.path.join(candidate, 'fake', '*/')))
        print(f'  Existing: {existing_real} real + {existing_fake} fake = {existing_real + existing_fake} videos')

        # Copy to working directory (Kaggle input is read-only)
        if not os.path.exists(output_path):
            print(f'  Copying to {output_path}...')
            t_copy = time.time()
            # Use cp -a for speed (preserves structure, uses hardlinks if possible)
            os.system(f'cp -a "{candidate}" "{output_path}"')
            print(f'  Copied in {time.time() - t_copy:.0f}s')
        else:
            print(f'  Output already exists at {output_path}, using as-is')

        restored = True
        break

# Also check if output_path already has content (e.g., notebook cell re-run)
if not restored and os.path.isdir(output_path):
    existing_dirs = 0
    for label in ('real', 'fake'):
        label_dir = os.path.join(output_path, label)
        if os.path.isdir(label_dir):
            existing_dirs += len([d for d in os.listdir(label_dir) if os.path.isdir(os.path.join(label_dir, d))])
    if existing_dirs > 0:
        print(f'\n=== RESUME: Found {existing_dirs} already processed videos in {output_path} ===')
        restored = True

if not restored:
    print('\n=== FRESH START: No partial output found ===')

# Step 5: Find DFD01 raw videos
print('\n=== Searching for DFD01 raw videos ===')
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}

dfd01_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    # Skip our own partial output
    if 'preprocessed' in root.lower():
        continue
    label_dirs = [d for d in dirs if d.lower() in ('real', 'fake', 'original', 'manipulated', 'actors', 'altered')]
    if not label_dirs:
        continue
    video_count = 0
    for ld in label_dirs:
        subdir = os.path.join(root, ld)
        if os.path.isdir(subdir):
            for sr, sd, sf in os.walk(subdir):
                video_count += sum(1 for f in sf if os.path.splitext(f)[1].lower() in VIDEO_EXTS)
    if video_count == 0:
        continue
    path_parts = root.replace('\\', '/').split('/')
    last_parts = '/'.join(path_parts[-2:]).lower()
    if 'dfd' in last_parts and 'dfdc' not in last_parts:
        dfd01_path = root
        print(f'Found DFD01: {root} ({video_count} videos)')
        break

if dfd01_path is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'preprocessed' in root.lower():
            continue
        if root.rstrip('/').endswith('DFD_01') or '/DFD_01/' in root:
            dfd01_path = root
            print(f'Found DFD01 (fallback): {root}')
            break

if dfd01_path is None:
    print('ERROR: DFD01 not found! Listing /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            print(f'{"  " * level}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    raise RuntimeError('DFD01 not found! Add dfdc-dataset-02-dfd-01 as input.')

# Step 6: Run preprocessing (with auto-resume)
print(f'\n{"="*60}')
print(f'PREPROCESSING DFD01 T={NUM_FRAMES}')
print(f'  Input:  {dfd01_path}')
print(f'  Output: {output_path}')
if restored:
    print(f'  MODE:   RESUME (skipping already processed)')
else:
    print(f'  MODE:   FRESH START')
print(f'{"="*60}\n')

t0 = time.time()

gpu_check = subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.is_available())'],
                           capture_output=True, text=True)
device = 'cuda' if gpu_check.stdout.strip() == 'True' else 'cpu'

cmd = [
    sys.executable, '-u', '/kaggle/working/project/preprocess_videos.py',
    dfd01_path, output_path,
    '--max-frames', str(NUM_FRAMES),
    '--output-size', str(OUTPUT_SIZE),
    '--min-face-confidence', str(MIN_FACE_CONF),
    '--min-detection-ratio', str(MIN_DETECTION_RATIO),
    '--detector-max-side', str(DETECTOR_MAX_SIDE),
    '--device', device,
]

print(f'Command: {" ".join(cmd)}')
proc = subprocess.Popen(cmd, cwd='/kaggle/working/project',
                        env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

elapsed = (time.time() - t0) / 60
print(f'\nDFD01 preprocessing done in {elapsed:.1f} min (exit code: {proc.returncode})')

# Step 7: Summary
import json
summary_path = os.path.join(output_path, 'summary.json')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        summary = json.load(f)
    print(f'\n--- DFD01 SUMMARY ---')
    print(f'Total videos: {summary["total_videos"]}')
    print(f'Saved:   {summary["saved_videos"]} (real={summary["real_saved"]}, fake={summary["fake_saved"]})')
    print(f'Dropped: {summary["dropped_videos"]}')
    print(f'Errors:  {summary["error_videos"]}')

# Step 8: Package
print(f'\nPackaging {output_name}.tar.gz...')
os.system(f'tar czf /kaggle/working/{output_name}.tar.gz -C /kaggle/working {output_name}/')
tar_size = os.path.getsize(f'/kaggle/working/{output_name}.tar.gz') / (1024**2)
print(f'\n{"="*60}')
print(f'DFD01 T={NUM_FRAMES} PREPROCESSING COMPLETE')
print(f'{"="*60}')
print(f'Output: {output_name}.tar.gz ({tar_size:.1f} MB)')
print('Upload as Kaggle dataset for multi-dataset training.')